In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import r2_score, mean_squared_error
import shap, warnings
warnings.filterwarnings("ignore")

# ---------- 0. 路径 ----------
DATA_FILE  = "Boring_scored.xlsx"         
UNSCORED    = "images.xlsx"
OUT_DIR    = Path("D:\Desktop\output_Boring2"); OUT_DIR.mkdir(exist_ok=True)

PREDICT_XLSX = OUT_DIR / "images_predicted_Boring.xlsx"
BSWARM_PNG   = OUT_DIR / "shap_beeswarm_Boring.png"
BAR_PNG      = OUT_DIR / "shap_bar_Boring.png"

# ---------- 1. 常量 ----------
TARGET = "Boring"
FEATS  = ["Road","Building","Pole Group","Indicator","Vegetation","Sky",
          "Person","Car","Motorcycle","Bicycle","Clothes","Trash Can",
          "Riverway","Signboard","Air Conditioner Condenser","Festival Elements"]
NOISE_MAX = 0.03
RF_PARAMS = dict(n_estimators=400, max_depth=9,
                 min_samples_leaf=5, max_features='sqrt',
                 random_state=42, n_jobs=-1)

# ---------- 2. 读取 & 微扰 ----------
df = pd.read_excel(DATA_FILE)
df[TARGET] = df[TARGET].round(5)

X = df[FEATS].copy()
rng = np.random.default_rng(42)
for col in FEATS:
    z = X[col] == 0
    if z.any():
        X.loc[z, col] += rng.uniform(0.01, NOISE_MAX, size=z.sum())

y = df[TARGET].values

# ---------- 3. 划分 & 随机森林 ----------
X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.20, random_state=42, shuffle=True)

rf = RandomForestRegressor(**RF_PARAMS).fit(X_tr, y_tr)

# ---------- 4. 单调校准 ----------
iso = IsotonicRegression(
        y_min=y.min(), y_max=y.max(),
        increasing=True, out_of_bounds="clip"
     ).fit(rf.predict(X), y)

# ---------- 5. 评估 ----------
def metr(t,p): return r2_score(t,p), mean_squared_error(t,p,squared=False)
cal_tr = iso.transform(rf.predict(X_tr)); cal_te = iso.transform(rf.predict(X_te))
r2_tr, rmse_tr = metr(y_tr, cal_tr)
r2_te, rmse_te = metr(y_te, cal_te)
r2_all, rmse_all = metr(y, iso.transform(rf.predict(X)))


print(f"Train   R²={r2_tr :.3f} | RMSE={rmse_tr :.3f}")
print(f"Test    R²={r2_te :.3f} | RMSE={rmse_te :.3f}")
print(f"Overall R²={r2_all:.3f} | RMSE={rmse_all:.3f}")

# ---------- 6. SHAP ----------
explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X)

plt.figure(figsize=(7,6), dpi=300)
shap.summary_plot(shap_values, X, feature_names=FEATS, show=False)
plt.title("SHAP Beeswarm – Boring")
plt.tight_layout(); plt.savefig(BSWARM_PNG, bbox_inches="tight",dpi=300); plt.close()

plt.figure(figsize=(7,6), dpi=300)
shap.summary_plot(shap_values, X, feature_names=FEATS,
                  plot_type="bar", show=False)
plt.title("Feature Importance – Boring")

plt.savefig(BAR_PNG, bbox_inches="tight",dpi=300); plt.close()

# ---------- 7. 预测未打分样本 ----------
df_new = pd.read_excel(UNSCORED)
X_new  = df_new[FEATS].copy()
for col in FEATS:
    z = X_new[col] == 0
    if z.any():
        X_new.loc[z, col] += rng.uniform(0.01, NOISE_MAX, size=z.sum())

df_new[TARGET] = iso.transform(rf.predict(X_new)).round(5)
df_new.to_excel(PREDICT_XLSX, index=False, float_format="%.5f")

print("✔ 预测文件:", PREDICT_XLSX)
print("✔ SHAP 图:", BSWARM_PNG, BAR_PNG)


Train   R²=0.947 | RMSE=0.119
Test    R²=0.867 | RMSE=0.183
Overall R²=0.932 | RMSE=0.134
✔ 预测文件: D:\Desktop\output_Boring2\images_predicted_Boring.xlsx
✔ SHAP 图: D:\Desktop\output_Boring2\shap_beeswarm_Boring.png D:\Desktop\output_Boring2\shap_bar_Boring.png
